Just run the cell below. Click on the output, not on the code part of the cell; that opens the whole code. If you accidentally open it, go `View -> Collapse Selected Code` to close it again.

In [207]:
group = 'rigi'

import re
from collections import Counter, defaultdict
from itertools import chain, repeat
from pathlib import Path

import pandas as pd
import rich
from rich.table import Table
from rich.text import Text


# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
from IPython.display import display_html, display
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)


########################
# READ ALL ACTIVE JOBS #


# We need to go with length and cut by lenght, because some entries may have spaces, some may be too long, etc.
jobs = !squeue -A {group} -O JobId:20,Name:20,UserName:20,State:20,TimeUsed:20,NumCPUs:20,QOS:20,NumNodes:20,GRES:20,RestartCnt:20,Reason:20
jobs = [[j[i*20:(i+1)*20].strip() for i in range(11)] for j in jobs]
jobs = pd.DataFrame(jobs[1:], columns=jobs[0])



##########################
# READ ALL WORKDIRS EVER #

# _xid_re = re.compile(r'(\d\d)(\d\d)_(\d\d)(\d\d)(\d\d)')
_xid_re = re.compile(r'\d\d\d\d_\d\d\d\d\d\d')
def extract_xid(name):
    if firstmatch := _xid_re.search(name):
        return firstmatch.group()
    return None


basedir = Path('/checkpoint/rigi/bv2/workdirs')
workdirs = [d.name for d in basedir.iterdir() if d.is_dir()]
wd_by_xid = {xid: wd for wd in workdirs if (xid := extract_xid(wd))}

#####################################
# SPLIT INTO CURRENT / RECENT / OLD #
# We do this split to add much more info to recent, and less to old.

hot_runs = {}
cold_runs = {}
for xid, wd in sorted(wd_by_xid.items(), reverse=True):
    states = Counter(jobs[jobs.NAME == xid].STATE)
    if states:
        hot_runs[xid] = {"states": states, "wd": wd}
    else:
        cold_runs[xid] = wd
frozen_runs = {xid: {"wd": cold_runs[xid]} for xid in list(cold_runs)[50:]}
cold_runs = {xid: {"wd": cold_runs[xid]} for xid in list(cold_runs)[:50]}

for xid, db in chain(zip(hot_runs.keys(), repeat(hot_runs)), zip(cold_runs.keys(), repeat(cold_runs))):
    launchinfo = basedir / db[xid]["wd"] / 'launchinfo.txt'
    if launchinfo.is_file():  # Launched with our sweep launcher
        db[xid]["wus"] = []
        for wuwd in (basedir / db[xid]["wd"]).iterdir():
            if wuwd.is_dir():
                db[xid]["wus"].append(wuwd)
        db[xid]["config"] = next(re.finditer(r"bv2/config/(.*?) ", launchinfo.read_text())).group(1)


#########################
# PREPARE VISUALIZATION #


tblH = Table(show_header=True, header_style="bold magenta", show_footer=True, footer_style="bold magenta", box=rich.box.HORIZONTALS)
tblH.add_column("xid", justify="left")
tblH.add_column("usr", justify="left")
tblH.add_column("states", justify="left")
tblH.add_column("wus", justify="right")
tblH.add_column("QoS", justify="left")
tblH.add_column("config", justify="left")

STATE_NAMES = {"RUNNING": Text("Run", "green"),
               "PENDING": Text("Pend", "yellow"),
               "REQUEUE_HOLD": Text("Hold", "red"),
               "COMPLETING": Text("End", "blue")}
# STATE_NAMES = {"RUNNING": "🏃", "PENDING": "⏳", "REQUEUE_HOLD": "🚫"}  # Sadly misaligns columns.

all_states = Counter()
for xid, info in hot_runs.items():
    all_states.update(info["states"])
    xjobs = jobs[jobs.NAME == xid]
    qos = ' '.join(xjobs.QOS.unique().tolist())
    users = ' '.join(xjobs.USER.unique().tolist())
    states = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in info["states"].most_common())
    tblH.add_row(xid, users, states, str(len(info["wus"])), qos, info["config"])

tblH.columns[2].footer = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in all_states.most_common())

tblC = Table(show_header=True, header_style="bold magenta", box=rich.box.HORIZONTALS)
tblC.add_column("xid", justify="left")
tblC.add_column("usr", justify="left")
tblC.add_column("work units", justify="left")
for xid, info in cold_runs.items():
    user = (basedir / info["wd"]).owner()
    tblC.add_row(xid, user, str(len(info["wus"])))

rich.print('Active runs:', tblH, 'Recent inactive runs:', tblC, f'Very old runs: [{len(frozen_runs)} not shown]')

Active runs:
 ────────────────────────────────────────────────────────────────────────────────────── 
  xid           usr    states                  wus   QoS              config            
 ────────────────────────────────────────────────────────────────────────────────────── 
  1119_113236   pplx   Run:4                     4   h100_rigi_high   synth_ocr.py      
  1118_160143   zhai   Run:6                     6   h200_lowest      finevision_8n.py  
  1118_160130   zhai   Pend:3 Run:2              6   h200_lowest      finevision_8n.py  
  1118_160032   zhai   Run:6                     6   h200_lowest      finevision_8n.py  
  1118_131059   zhai   Run:5 Hold:4              9   h200_lowest      finevision_8n.py  
  1118_130956   zhai   Run:9                     9   h200_lowest      finevision_8n.py  
  1118_130432   zhai   Run:3 Pend:1 Hold:1       9   h200_lowest      finevision_8n.py  
  1117_163432   qkv    Run:15 Hold:2           120   h100_lowest      code.py           
  1117_134621   qkv    Run:5                    40   h100_rigi_high   finevision.py     
  1114_221425   qkv    Hold:55                 235   h100_lowest      code.py           
 ────────────────────────────────────────────────────────────────────────────────────── 
                       Hold:62 Run:55 Pend:4                                            
 ────────────────────────────────────────────────────────────────────────────────────── 
Recent inactive runs:
 ───────────────────────────────── 
  xid           usr    work units  
 ───────────────────────────────── 
  1119_112304   pplx   0           
  1118_135300   zhai   3           
  1118_135236   zhai   0           
  1118_135207   zhai   1           
  1117_164507   zhai   2           
  1117_164452   zhai   2           
  1117_154745   zhai   2           
  1117_154525   zhai   2           
  1117_154217   zhai   2           
  1117_154143   zhai   2           
  1117_153042   zhai   2           
  1117_152839   zhai   0           
  1117_141721   zhai   24          
  1117_092405   zhai   72          
  1114_225004   qkv    40          
  1114_223757   qkv    3           
  1114_222316   zhai   0           
  1114_220730   qkv    9           
  1114_215835   zhai   0           
  1114_210506   qkv    1           
  1114_165903   qkv    1           
  1114_161036   qkv    1           
  1114_160932   qkv    1           
  1114_151320   qkv    1           
  1114_115349   qkv    1           
  1114_113742   qkv    0           
  1114_111845   qkv    1           
  1114_110459   qkv    1           
  1114_105722   qkv    1           
  1114_105527   qkv    0           
  1114_101254   qkv    0           
  1114_100333   qkv    1           
  1114_095538   qkv    1           
  1114_095054   qkv    1           
  1114_094133   qkv    1           
  1113_195249   qkv    1           
  1113_195241   qkv    1           
  1113_195233   qkv    1           
  1113_170331   qkv    1           
  1113_170323   qkv    1           
  1113_170315   qkv    0           
  1113_162203   qkv    1           
  1113_162150   qkv    1           
  1113_162139   qkv    1           
  1113_161505   qkv    1           
  1113_161451   qkv    1           
  1113_161438   qkv    1           
  1113_160432   qkv    1           
  1113_160421   qkv    1           
  1113_160410   qkv    1           
 ───────────────────────────────── 
Very old runs: [158 not shown]

# Tmp/dev

This is a place to dig deeper or figure out some things. The useful variables are: `jobs` (from `squeue` command), `hot_runs`, and `workdirs` (or sth like `basedir / workdirs[0]`).

In [82]:
jobs.query('NAME == "1114_221425"')

,JOBID,NAME,USER,STATE,TIME,CPUS,QOS,NODES,TRES_PER_NODE,RESTART_COUNT,REASON
0,942235,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,4,Resources
1,942053,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,6,Priority
2,942054,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,6,Priority
3,942240,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
4,942241,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
5,942242,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
6,942243,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
7,942244,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
8,942245,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
9,942246,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
